# Fraud – Hakem yorumları için ek deneyler

Bu defter yalnızca hakemlerin istediği ek analizleri içerir. Alt örnekleme kodları mevcut Fraud defterlerinizle birebir aynıdır.

| Hücre | İçerik | Hakem yorumu |
|---|---|---|
| 4 | Sızıntısız SVM/QSVM ve k-NN/QKNN (katman içi ön işleme), kernel hizalama | H2-Y3, Y4, Y14 |
| 5 | Saf Klasik ANN, Klasik Kontrol, QNN 2/4/8 – 200 epoch, 30/100/200 kontrol noktaları | H2-Y5, Y11 |
| 6 | Klasik AE (Sığ, Derin) ve QAE – PR-AUC eklendi | H2-Y12 |
| 7 | Kümeleme – tüm modellerde Hungarian, katman içi kuantum mesafe matrisi | H2-Y13, Y3 |
| 8 | Eşleştirilmiş t-testleri (Bonferroni) ve sonuçların kaydı | H2-Y4 |

**Çalıştırma:** Tümünü çalıştırın; veri seti otomatik indirilir ve orijinal dosyayla aynı olduğu doğrulanır (indirme başarısız olursa elle yükleme istenir). En uzun hücreler 5 (8 kübitlik QNN, 200 epoch) ve 4. Mevcut defterlerinizle aynı çalışma zamanı türünü seçin.

**Bitince:** `fraud_ek_sonuclar.zip` indirilir; bu dosyayı ve ekran çıktısını gönderin.


In [ ]:
!pip install -q pennylane

In [ ]:
import json, os, platform, sys, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.optimize import linear_sum_assignment
from scipy.stats import ttest_rel, wilcoxon, spearmanr, pearsonr
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix,
                             adjusted_rand_score, normalized_mutual_info_score, silhouette_score)
from sklearn.metrics.pairwise import rbf_kernel, euclidean_distances
warnings.filterwarnings("ignore")
RANDOM_SEED = 42; N_QUBITS = 4; N_SPLITS = 5
np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
ALL = {}

def ms(a, scale=1.0, nd=2):
    a = np.asarray(a, float) * scale
    return f"{a.mean():.{nd}f} ± {a.std():.{nd}f}".replace(".", ",")

def environment():
    import sklearn, scipy
    info = {"Python": sys.version.split()[0], "OS": platform.platform(), "NumPy": np.__version__, "scikit-learn": sklearn.__version__,
            "SciPy": scipy.__version__, "PennyLane": qml.__version__, "PyTorch": torch.__version__,
            "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Yok (CPU)", "CPU çekirdek": os.cpu_count()}
    try:
        info["RAM (GB)"] = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 2)
        with open("/proc/cpuinfo") as f:
            for line in f:
                if "model name" in line: info["CPU modeli"] = line.split(":")[1].strip(); break
    except Exception: pass
    return info

def folds_of(X, y):
    return list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED).split(X, y))

def angle_pipeline(X_tr, X_te, n_comp=N_QUBITS):
    """StandardScaler -> PCA(4) -> MinMax[-pi, pi]; yalnızca eğitim katmanında fit (sızıntısız)."""
    sc = StandardScaler().fit(X_tr)
    pca = PCA(n_components=n_comp, random_state=RANDOM_SEED).fit(sc.transform(X_tr))
    mm = MinMaxScaler(feature_range=(-np.pi, np.pi)).fit(pca.transform(sc.transform(X_tr)))
    f = lambda Z: mm.transform(pca.transform(sc.transform(Z)))
    return f(X_tr), f(X_te), pca.explained_variance_ratio_

def hungarian_align(y_true, clusters, k):
    """Küme etiketlerini Hungarian (Kuhn-Munkres) optimal ataması ile gerçek sınıflara eşler (Hakem #2 Yorum #13)."""
    cm = confusion_matrix(y_true, clusters, labels=range(k))
    r, c = linear_sum_assignment(cm.max() - cm); mp = {cc: rr for rr, cc in zip(r, c)}
    return np.array([mp.get(v, v) for v in clusters])

def paired_tests(pairs, metric_name="F1"):
    """Hakem #2 Yorum #4: katlama bazında eşleştirilmiş t-testi + Bonferroni. pairs: [(ad_A, ad_B, katlamalar_A, katlamalar_B), ...]
    Wilcoxon yalnızca bilgi amaçlı (N=5'te iki yönlü en küçük p = 0,0625)."""
    pairs = [p for p in pairs if p[2] is not None and p[3] is not None and len(p[2]) == len(p[3]) == N_SPLITS]
    m = len(pairs); rows = []
    for a, b, xa, xb in pairs:
        xa, xb = np.asarray(xa, float) * 100, np.asarray(xb, float) * 100; d = xa - xb
        if np.allclose(d, 0): t, p, w = 0.0, 1.0, 1.0
        else:
            t, p = ttest_rel(xa, xb)
            try: w = wilcoxon(xa, xb).pvalue
            except Exception: w = float("nan")
        rows.append({"A": a, "B": b, "fark_ort": float(d.mean()), "fark_sd": float(d.std(ddof=1)), "t": float(t), "p": float(p),
                     "p_bonferroni": float(min(1.0, p * m)), "wilcoxon_p": float(w), "anlamli": bool(min(1.0, p * m) < 0.05)})
    print("\n" + "=" * 130); print(f"EŞLEŞTİRİLMİŞ t-TESTİ ({metric_name}, N = 5 katlama, Bonferroni m = {m})"); print("=" * 130)
    print(f"{'Model A':<32} | {'Model B':<32} | {'Fark (A-B)':<15} | {'t':>7} | {'p':>7} | {'p (Bonf.)':>9} | {'Wilcoxon p':>10} | Anlamlı")
    print("-" * 130)
    for r in rows:
        print(f"{r['A']:<32} | {r['B']:<32} | {r['fark_ort']:6.2f} ± {r['fark_sd']:5.2f}  | {r['t']:7.2f} | {r['p']:7.4f} | {r['p_bonferroni']:9.4f} | {r['wilcoxon_p']:10.4f} | {'Evet' if r['anlamli'] else 'Hayır'}")
    print("=" * 130)
    return rows

def to_py(o):
    if isinstance(o, dict): return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    return o


In [ ]:
# ---------- KUANTUM SADAKAT MATRİSLERİ (katman içinde, sızıntısız) ----------
# Mevcut defterlerinizle aynı devreler: AngleEmbedding (varsayılan R_X), Adjoint 4 kübit ve Swap-Test 9 kübit, shots=None.
dev_adj = qml.device("default.qubit", wires=N_QUBITS)
@qml.qnode(dev_adj, interface="autograd")
def adjoint_qnode(x1, x2):
    qml.AngleEmbedding(x1, wires=range(N_QUBITS)); qml.adjoint(qml.AngleEmbedding)(x2, wires=range(N_QUBITS))
    return qml.probs(wires=range(N_QUBITS))
dev_swap = qml.device("default.qubit", wires=2 * N_QUBITS + 1)
@qml.qnode(dev_swap, interface="autograd")
def swap_qnode(x1, x2):
    qml.Hadamard(wires=0)
    qml.AngleEmbedding(x1, wires=range(1, N_QUBITS + 1)); qml.AngleEmbedding(x2, wires=range(N_QUBITS + 1, 2 * N_QUBITS + 1))
    for i in range(N_QUBITS): qml.CSWAP(wires=[0, i + 1, i + N_QUBITS + 1])
    qml.Hadamard(wires=0); return qml.expval(qml.PauliZ(0))
def fidelity(kind, a, b):
    return float(adjoint_qnode(a, b)[0]) if kind == "adjoint" else float(np.clip(swap_qnode(a, b), 0.0, 1.0))
def fidelity_matrix(kind, A, B=None):
    sym = B is None; B = A if sym else B; F = np.zeros((len(A), len(B)))
    for i in range(len(A)):
        for j in range(len(B)):
            if sym and j < i: F[i, j] = F[j, i]
            elif sym and i == j: F[i, j] = 1.0
            else: F[i, j] = fidelity(kind, A[i], B[j])
    return F
qdist = lambda F: np.sqrt(np.maximum(0.0, 2.0 - 2.0 * np.clip(F, 0.0, 1.0)))
def compute_kernels(X, y, kinds=("adjoint", "swap"), label=""):
    folds = folds_of(X, y); ANG, KER = [], {}
    for k, (tr, te) in enumerate(folds, 1):
        A_tr, A_te, var = angle_pipeline(X[tr], X[te]); ANG.append((A_tr, A_te, var))
        for kind in kinds:
            t0 = time.time(); Ftr = fidelity_matrix(kind, A_tr); t1 = time.time() - t0
            t0 = time.time(); Fte = fidelity_matrix(kind, A_te, A_tr); t2 = time.time() - t0
            KER[(k, kind)] = {"tr": Ftr, "te": Fte, "t": t1 + t2}
            print(f"{label} Fold {k} {kind:<7}: {t1+t2:6.1f} sn")
    return folds, ANG, KER

# ---------- SIZINTISIZ SVM / QSVM ve k-NN / QKNN (Hakem #2 Yorum #3, katman bazlı sonuçlar Yorum #4 için) ----------
# Mevcut defterlerde CV, tüm veriye fit edilen ölçekleyici/PCA ile hesaplanıyordu; burada her katmanda yalnızca eğitim verisiyle fit edilir.
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
def run_supervised(X, y, folds, ANG, KER, n_classes, title, k_neighbors=5):
    MODELS = ["Klasik SVM (Linear)", "Klasik SVM (RBF)", "QSVM (Adjoint Kernel)", "QSVM (Swap Test)",
              "Klasik k-NN (Öklid)", "QKNN (Adjoint Kernel)", "QKNN (Swap Test)"]
    R = {m: {"acc": [], "f1": [], "f1_macro": [], "auc": [], "time": []} for m in MODELS}
    for k, ((tr, te), (A_tr, A_te, _)) in enumerate(zip(folds, ANG), 1):
        y_tr, y_te = y[tr], y[te]
        for m in MODELS:
            kind = "adjoint" if "Adjoint" in m else "swap"; t0 = time.time()
            if m.startswith("Klasik SVM"):
                clf = SVC(kernel="linear" if "Linear" in m else "rbf", C=1.0, probability=True, random_state=RANDOM_SEED).fit(A_tr, y_tr)
                pred, prob = clf.predict(A_te), clf.predict_proba(A_te)
            elif m.startswith("QSVM"):
                clf = SVC(kernel="precomputed", C=1.0, probability=True, random_state=RANDOM_SEED).fit(KER[(k, kind)]["tr"], y_tr)
                pred, prob = clf.predict(KER[(k, kind)]["te"]), clf.predict_proba(KER[(k, kind)]["te"])
            elif m.startswith("Klasik k-NN"):
                clf = KNeighborsClassifier(n_neighbors=k_neighbors, metric="euclidean").fit(A_tr, y_tr)
                pred, prob = clf.predict(A_te), clf.predict_proba(A_te)
            else:
                Dtr, Dte = qdist(KER[(k, kind)]["tr"]), qdist(KER[(k, kind)]["te"]); np.fill_diagonal(Dtr, 0.0)
                clf = KNeighborsClassifier(n_neighbors=k_neighbors, metric="precomputed").fit(Dtr, y_tr)
                pred, prob = clf.predict(Dte), clf.predict_proba(Dte)
            el = time.time() - t0 + (0.0 if m.startswith("Klasik") else KER[(k, kind)]["t"])
            s = R[m]; s["acc"].append(accuracy_score(y_te, pred)); s["time"].append(el)
            if n_classes == 2:
                s["f1"].append(f1_score(y_te, pred, zero_division=0)); s["f1_macro"].append(f1_score(y_te, pred, average="macro", zero_division=0))
                s["auc"].append(roc_auc_score(y_te, prob[:, 1]))
            else:
                s["f1"].append(f1_score(y_te, pred, average="weighted", zero_division=0)); s["f1_macro"].append(f1_score(y_te, pred, average="macro", zero_division=0))
                s["auc"].append(roc_auc_score(y_te, prob, multi_class="ovr", average="macro"))
    f1lbl = "F1 Binary" if n_classes == 2 else "F1 Weighted"
    print("\n" + "=" * 128); print(f"{title} – SIZINTISIZ SVM / k-NN (5-katlı CV, Mean ± SD; süre: katman başına)"); print("=" * 128)
    print(f"{'Model':<24} | {'Doğruluk (%)':<15} | {f1lbl+' (%)':<15} | {'F1 Macro (%)':<15} | {'ROC-AUC':<15} | Süre (sn)"); print("-" * 128)
    for m in MODELS:
        s = R[m]; print(f"{m:<24} | {ms(s['acc'],100):<15} | {ms(s['f1'],100):<15} | {ms(s['f1_macro'],100):<15} | {ms(s['auc'],1,3):<15} | {ms(s['time'],1,3)}")
    print("=" * 128)
    var = np.array([a[2] for a in ANG]); print(f"PCA kümülatif varyans (katman ort.): %{ms(var.sum(1),100)}")
    return R

# ---------- KÜMELEME: tüm modellerde Hungarian eşleme, katman içi kuantum mesafe matrisi (Hakem #2 Yorum #13 ve #3) ----------
class QuantumKMeans:
    """Mevcut defterinizdeki 9 kübitlik Swap-Test Q-Means ile aynı (en fazla 5 iterasyon)."""
    def __init__(self, n_clusters=2, max_iter=5, random_state=RANDOM_SEED):
        self.k, self.max_iter, self.rs = n_clusters, max_iter, random_state
    def _d(self, x, c): return float(qdist(np.clip(swap_qnode(x, c), 0, 1)))
    def fit(self, X):
        np.random.seed(self.rs); self.centroids = X[np.random.choice(len(X), self.k, replace=False)].copy(); labels = np.zeros(len(X))
        for _ in range(self.max_iter):
            new = np.array([np.argmin([self._d(x, c) for c in self.centroids]) for x in X])
            if np.all(new == labels): break
            labels = new
            for j in range(self.k):
                if np.any(labels == j): self.centroids[j] = X[labels == j].mean(axis=0)
        return self
    def predict(self, X): return np.array([np.argmin([self._d(x, c) for c in self.centroids]) for x in X]).astype(int)

def run_clustering(X, y, n_clusters, title):
    folds = folds_of(X, y); MODELS = ["Klasik K-Means", "Kuantum Çekirdek Kümeleme", "Swap-Test Q-Means"]
    R = {m: {k: [] for k in ("acc", "f1", "f1_macro", "ari", "nmi", "sil", "time")} for m in MODELS}
    avg = "binary" if n_clusters == 2 else "weighted"
    def ev(m, yt, c, Xe, el):
        ya = hungarian_align(yt, c, n_clusters); s = R[m]
        s["acc"].append(accuracy_score(yt, ya)); s["f1"].append(f1_score(yt, ya, average=avg, zero_division=0))
        s["f1_macro"].append(f1_score(yt, ya, average="macro", zero_division=0)); s["ari"].append(adjusted_rand_score(yt, c))
        s["nmi"].append(normalized_mutual_info_score(yt, c)); s["sil"].append(silhouette_score(Xe, c) if len(set(c)) > 1 else 0.0); s["time"].append(el)
    for k, (tr, te) in enumerate(folds, 1):
        A_tr, A_te, _ = angle_pipeline(X[tr], X[te]); y_te = y[te]
        t0 = time.time(); c = KMeans(n_clusters=n_clusters, random_state=RANDOM_SEED, n_init=10).fit(A_tr).predict(A_te); ev(MODELS[0], y_te, c, A_te, time.time() - t0)
        t0 = time.time()
        Dtr = qdist(fidelity_matrix("adjoint", A_tr)); Dte = qdist(fidelity_matrix("adjoint", A_te, A_tr)); np.fill_diagonal(Dtr, 0.0)
        kq = KMeans(n_clusters=n_clusters, random_state=RANDOM_SEED, n_init=10).fit(Dtr)
        med = [int(np.argmin(np.linalg.norm(Dtr - cc, axis=1))) for cc in kq.cluster_centers_]
        c = np.argmin(Dte[:, med], axis=1); ev(MODELS[1], y_te, c, A_te, time.time() - t0)
        t0 = time.time(); c = QuantumKMeans(n_clusters).fit(A_tr).predict(A_te); ev(MODELS[2], y_te, c, A_te, time.time() - t0)
        print(f"Fold {k}/5 tamamlandı")
    print("\n" + "=" * 150); print(f"{title} – KÜMELEME (k={n_clusters}, 5-katlı CV, Mean ± SD, Hungarian eşleme)"); print("=" * 150)
    print(f"{'Model':<26} | {'Doğruluk (%)':<15} | {'F1 (%)':<15} | {'F1 Macro (%)':<15} | {'ARI':<14} | {'NMI':<14} | {'Silüet':<14} | Süre (sn)"); print("-" * 150)
    for m in MODELS:
        s = R[m]; print(f"{m:<26} | {ms(s['acc'],100):<15} | {ms(s['f1'],100):<15} | {ms(s['f1_macro'],100):<15} | {ms(s['ari'],1,3):<14} | {ms(s['nmi'],1,3):<14} | {ms(s['sil'],1,3):<14} | {ms(s['time'],1,2)}")
    print("=" * 150)
    return R

# ---------- KUANTUM ÇEKİRDEK – RBF HİZALAMASI ve MESAFE KORELASYONU (Hakem #2 Yorum #14, Çizelge 24) ----------
# AngleEmbedding (R_X) dolanıklık içermediğinden sadakat analitik olarak k(x,x') = prod cos^2((x_i - x'_i)/2) şeklindedir (Adjoint devresiyle aynı).
def run_alignment(X, y, title):
    def qk(A, B): d = A[:, None, :] - B[None, :, :]; return np.prod(np.cos(d / 2) ** 2, axis=2)
    def align(K1, K2): return np.sum(K1 * K2) / np.sqrt(np.sum(K1 * K1) * np.sum(K2 * K2))
    def calign(K1, K2): n = len(K1); H = np.eye(n) - 1 / n; return align(H @ K1 @ H, H @ K2 @ H)
    r = {k: [] for k in ["spearman", "pearson", "align_rbf_scale", "centered_align_rbf_scale", "align_rbf_g025"]}
    for tr, te in folds_of(X, y):
        A, _, _ = angle_pipeline(X[tr], X[te]); Kq = qk(A, A); Dq = np.sqrt(np.maximum(0, 2 - 2 * Kq)); De = euclidean_distances(A)
        iu = np.triu_indices(len(A), 1)
        r["spearman"].append(spearmanr(Dq[iu], De[iu]).correlation); r["pearson"].append(pearsonr(Dq[iu], De[iu])[0])
        g = 1 / (A.shape[1] * A.var()); Kr = rbf_kernel(A, gamma=g)
        r["align_rbf_scale"].append(align(Kq, Kr)); r["centered_align_rbf_scale"].append(calign(Kq, Kr)); r["align_rbf_g025"].append(align(Kq, rbf_kernel(A, gamma=0.25)))
    print(f"\n{title} – Çizelge 24 satırı:")
    for k, v in r.items(): print(f"  {k:<26}: {ms(v,1,3)}")
    return r

# ---------- YAPAY SİNİR AĞLARI: Saf Klasik ANN, Klasik Kontrol, Hibrit QNN (Hakem #2 Yorum #11 ve #5) ----------
# Eşit koşullar: aynı katmanlar, aynı optimize edici/öğrenme oranı/mini-yığın, erken durdurma yok, tüm modellerde 200 epoch.
# 30/100/200. epoch sonuçları AYNI eğitimin kontrol noktalarıdır. Klasik Kontrol: QNN-4 ile aynı giriş ve darboğaz boyutu,
# yaklaşık aynı parametre sayısı; kuantum katman yerine Linear(4,4)+tanh klasik doğrusal olmayan darboğaz.
# NOT: LR, BATCH_SIZE ve EMBED_ROT değerlerini mevcut ANN/QNN defterinizdekilerle AYNI yapın (karşılaştırılabilirlik için).
LR, BATCH_SIZE, MAX_EPOCHS, CHECKPOINTS, EMBED_ROT = 0.002, 32, 200, (30, 100, 200), "Y"

def make_qlayer(n_qubits, n_layers=2):
    qdev = qml.device("default.qubit", wires=n_qubits)
    @qml.qnode(qdev, interface="torch", diff_method="backprop")
    def qnode(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation=EMBED_ROT)
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]
    return qml.qnn.TorchLayer(qnode, {"weights": (n_layers, n_qubits, 3)})

class HybridQNN(nn.Module):
    def __init__(self, d_in, n_qubits, n_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in, 16); self.dropout = nn.Dropout(0.3)
        self.fc_bottleneck = nn.Linear(16, n_qubits); self.qlayer = make_qlayer(n_qubits); self.fc_out = nn.Linear(n_qubits, n_out)
    def forward(self, x):
        x = self.dropout(torch.relu(self.fc1(x))); x = torch.tanh(torch.relu(self.fc_bottleneck(x))) * np.pi
        return self.fc_out(self.qlayer(x))

class ClassicalControl(nn.Module):
    def __init__(self, d_in, n_bottleneck, n_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in, 16); self.dropout = nn.Dropout(0.3)
        self.fc_bottleneck = nn.Linear(16, n_bottleneck); self.classical_layer = nn.Linear(n_bottleneck, n_bottleneck)
        self.fc_out = nn.Linear(n_bottleneck, n_out)
    def forward(self, x):
        x = self.dropout(torch.relu(self.fc1(x))); x = torch.tanh(torch.relu(self.fc_bottleneck(x))) * np.pi
        return self.fc_out(torch.tanh(self.classical_layer(x)))

class PureClassicalANN(nn.Module):
    def __init__(self, d_in, n_out, bottleneck_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(d_in, 16); self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(16, bottleneck_dim); self.fc3 = nn.Linear(bottleneck_dim, bottleneck_dim); self.fc_out = nn.Linear(bottleneck_dim, n_out)
    def forward(self, x):
        x = self.dropout(torch.relu(self.fc1(x))); x = torch.relu(self.fc3(torch.relu(self.fc2(x)))); return self.fc_out(x)

def count_params(m):
    return (sum(p.numel() for p in m.parameters() if p.requires_grad),
            sum(p.numel() for n, p in m.named_parameters() if p.requires_grad and n.startswith("qlayer")))

def train_eval(builder, X, y, tr, te, seed, n_classes):
    torch.manual_seed(seed); np.random.seed(seed)
    sc = MinMaxScaler(feature_range=(-1, 1)).fit(X[tr])          # yalnızca eğitim katmanında fit
    X_tr = torch.tensor(sc.transform(X[tr]), dtype=torch.float32); X_te = torch.tensor(sc.transform(X[te]), dtype=torch.float32)
    binary = n_classes == 2
    y_tr = torch.tensor(y[tr], dtype=torch.float32).unsqueeze(1) if binary else torch.tensor(y[tr], dtype=torch.long); y_te = y[te]
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    model = builder(); crit = nn.BCEWithLogitsLoss() if binary else nn.CrossEntropyLoss(); opt = optim.Adam(model.parameters(), lr=LR)
    ck, t_train = {}, 0.0
    for ep in range(1, MAX_EPOCHS + 1):
        t0 = time.time(); model.train()
        for bx, by in loader:
            opt.zero_grad(); loss = crit(model(bx), by); loss.backward(); opt.step()
        t_train += time.time() - t0
        if ep in CHECKPOINTS:
            model.eval()
            with torch.no_grad(): out = model(X_te)
            if binary:
                prob = torch.sigmoid(out).numpy().ravel(); pred = (prob > 0.5).astype(int)
                e = {"acc": accuracy_score(y_te, pred), "f1": f1_score(y_te, pred, zero_division=0),
                     "auc": roc_auc_score(y_te, prob) if len(np.unique(prob)) > 1 else 0.5}
            else:
                prob = torch.softmax(out, dim=1).numpy(); pred = prob.argmax(1)
                e = {"acc": accuracy_score(y_te, pred), "f1": f1_score(y_te, pred, average="weighted", zero_division=0),
                     "f1_macro": f1_score(y_te, pred, average="macro", zero_division=0),
                     "auc": roc_auc_score(y_te, prob, multi_class="ovr", average="macro")}
            e["train_time"] = t_train; e["tek_sinif_tahmini"] = bool(len(np.unique(pred)) == 1); ck[ep] = e
    return ck

def run_nn_block(X, y, n_classes, title):
    d_in = X.shape[1]; n_out = 1 if n_classes == 2 else n_classes; folds = folds_of(X, y)
    MODELS = {"Saf Klasik ANN": lambda: PureClassicalANN(d_in, n_out, 2),
              "Klasik Kontrol (4 boyutlu darboğaz)": lambda: ClassicalControl(d_in, 4, n_out),
              "Hibrit QNN (2 Kübit)": lambda: HybridQNN(d_in, 2, n_out), "Hibrit QNN (4 Kübit)": lambda: HybridQNN(d_in, 4, n_out),
              "Hibrit QNN (8 Kübit)": lambda: HybridQNN(d_in, 8, n_out)}
    res = {}
    for name, b in MODELS.items():
        t0 = time.time(); runs = [train_eval(b, X, y, tr, te, RANDOM_SEED + k, n_classes) for k, (tr, te) in enumerate(folds, 1)]
        tot, q = count_params(b())
        res[name] = {"params_total": tot, "params_quantum": q, "checkpoints": {ep: [r[ep] for r in runs] for ep in CHECKPOINTS}}
        print(f"[{name}] tamamlandı ({(time.time()-t0)/60:.1f} dk)")
    f1lbl = "F1 (Binary)" if n_classes == 2 else "F1 (Weighted)"
    print("\n" + "=" * 150); print(f"{title} – ANN / KLASİK KONTROL / QNN (5-katlı CV, Mean ± SD; süre: katman başına eğitim)"); print("=" * 150)
    print(f"{'Model':<36} | {'Param (top/kuantum)':<19} | {'Epoch':<5} | {'Doğruluk (%)':<15} | {f1lbl+' (%)':<16} | {'ROC-AUC':<14} | {'Süre (sn)':<12} | Tek sınıf")
    print("-" * 150)
    for name, s in res.items():
        for ep in CHECKPOINTS:
            r = s["checkpoints"][ep]; g = lambda k: [x[k] for x in r]
            print(f"{name:<36} | {str(s['params_total'])+' / '+str(s['params_quantum']):<19} | {ep:<5} | {ms(g('acc'),100):<15} | {ms(g('f1'),100):<16} | {ms(g('auc'),1,3):<14} | {ms(g('train_time'),1,1):<12} | {sum(g('tek_sinif_tahmini'))}/5")
    print("=" * 150)
    return res

# ---------- ANOMALİ TESPİTİ: Klasik AE (Sığ, Derin) ve QAE, PR-AUC eklenmiş (Hakem #2 Yorum #12) ----------
# Mevcut defterinizle aynı protokol: 400 normal + 100 sahtekarlık, Time ve Class çıkarılır (29 öznitelik), tabakalı 5 katlama,
# ölçekleme/PCA/MinMax yalnızca eğitim katmanında fit, eğitim yalnızca eğitim katmanındaki normal örneklerle,
# eşik = eğitimdeki normal skorların 95. persentili (test etiketleri kullanılmaz). Tüm epoch değerleri aynı eğitimin kontrol noktalarıdır.
AE_CK, AE_MAX, LR_AE, LR_QAE = (20, 30, 50, 100), 100, 0.005, 0.02
class ClassicalAE(nn.Module):
    def __init__(self, deep):
        super().__init__()
        self.net = (nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2), nn.ReLU(), nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 4))
                    if deep else nn.Sequential(nn.Linear(4, 2), nn.ReLU(), nn.Linear(2, 4)))
    def forward(self, x): return self.net(x)
    def loss(self, x): return nn.functional.mse_loss(self(x), x)
    def score(self, x):
        with torch.no_grad(): return torch.mean((x - self(x)) ** 2, dim=1).numpy()
qae_dev = qml.device("default.qubit", wires=N_QUBITS)
@qml.qnode(qae_dev, interface="torch", diff_method="backprop")
def qae_qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS)); qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(2)), qml.expval(qml.PauliZ(3))]
class QuantumAE(nn.Module):
    def __init__(self):
        super().__init__(); self.qlayer = qml.qnn.TorchLayer(qae_qnode, {"weights": (2, N_QUBITS, 3)})
    def _s(self, x): z = self.qlayer(x); return 1.0 - (z[:, 0] + z[:, 1] + 2.0) / 4.0
    def loss(self, x): return torch.mean(self._s(x))
    def score(self, x):
        with torch.no_grad(): return self._s(x).numpy()
def run_autoencoders(X, y, title):
    folds = folds_of(X, y)
    MODELS = {"Klasik AE (Sığ 4-2-4)": (lambda: ClassicalAE(False), LR_AE), "Klasik AE (Derin 4-8-2-8-4)": (lambda: ClassicalAE(True), LR_AE),
              "Kuantum QAE (2 Katman)": (lambda: QuantumAE(), LR_QAE)}
    R = {m: {ep: {k: [] for k in ("auc", "pr_auc", "acc", "f1", "train_time")} for ep in AE_CK} for m in MODELS}
    for name, (builder, lr) in MODELS.items():
        for k, (tr, te) in enumerate(folds, 1):
            torch.manual_seed(RANDOM_SEED + k)
            A_tr, A_te, _ = angle_pipeline(X[tr], X[te])                 # mevcut defterle aynı: tüm eğitim katmanında fit
            x_trn = torch.tensor(A_tr[y[tr] == 0], dtype=torch.float32); x_te = torch.tensor(A_te, dtype=torch.float32)
            loader = DataLoader(x_trn, batch_size=32, shuffle=True); model = builder(); opt = optim.Adam(model.parameters(), lr=lr); tt = 0.0
            for ep in range(1, AE_MAX + 1):
                t0 = time.time(); model.train()
                for bx in loader: opt.zero_grad(); l = model.loss(bx); l.backward(); opt.step()
                tt += time.time() - t0
                if ep in AE_CK:
                    model.eval(); thr = np.percentile(model.score(x_trn), 95); s = model.score(x_te); pred = (s > thr).astype(int); yt = y[te]; r = R[name][ep]
                    r["auc"].append(roc_auc_score(yt, s)); r["pr_auc"].append(average_precision_score(yt, s))
                    r["acc"].append(accuracy_score(yt, pred)); r["f1"].append(f1_score(yt, pred, zero_division=0)); r["train_time"].append(tt)
        print(f"[{name}] tamamlandı")
    print("\n" + "=" * 136); print(f"{title} – ANOMALİ TESPİTİ (5-katlı CV, Mean ± SD; pozitif sınıf: sahtekarlık; eşik: eğitim normal 95. persentil)"); print("=" * 136)
    print(f"{'Model':<30} | {'Epoch':<5} | {'ROC-AUC (%)':<15} | {'PR-AUC (%)':<15} | {'Doğruluk (%)':<15} | {'F1 (%)':<15} | Süre (sn)"); print("-" * 136)
    for name in MODELS:
        for ep in AE_CK:
            r = R[name][ep]; print(f"{name:<30} | {ep:<5} | {ms(r['auc'],100):<15} | {ms(r['pr_auc'],100):<15} | {ms(r['acc'],100):<15} | {ms(r['f1'],100):<15} | {ms(r['train_time'],1,2)}")
    print("=" * 136)
    return R


In [ ]:
# ======================================================================================
# FRAUD – HAKEM YORUMLARI İÇİN EK DENEYLER
#   Yorum #3  : Sızıntısız SVM/QSVM ve k-NN/QKNN (katman içinde fit)
#   Yorum #4  : Katman bazlı sonuçlar + eşleştirilmiş t-testi (Bonferroni)
#   Yorum #5/#11: Eşit bütçe (200 epoch) + parametre eşlemeli Klasik Kontrol modeli
#   Yorum #12 : Anomali tespitinde PR-AUC
#   Yorum #13 : Kümelemede tüm modeller için Hungarian eşleme, katman içi kuantum mesafe matrisi
#   Yorum #14 : Kuantum çekirdek – RBF hizalaması (Çizelge 24 Fraud satırı)
# Alt örnekleme kodları mevcut defterlerinizle birebir aynıdır (random_state = 42).
# ======================================================================================
import os
# Veri seti otomatik indirilir (elle yükleme gerekmez). Sırasıyla denenir:
#   1) TensorFlow'un herkese açık kopyası (Kaggle "Credit Card Fraud Detection" dosyasının aynısı)
#   2) kagglehub (mlg-ulb/creditcardfraud)
#   3) Colab'a elle yükleme
CSV = "creditcard.csv"
if not os.path.exists(CSV):
    try:
        pd.read_csv("https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv").to_csv(CSV, index=False)
        print("İndirildi: TensorFlow kopyası")
    except Exception as e:
        print("TensorFlow kopyası indirilemedi:", e)
if not os.path.exists(CSV):
    try:
        import subprocess; subprocess.run(["pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub, shutil
        path = kagglehub.dataset_download("mlg-ulb/creditcardfraud"); shutil.copy(os.path.join(path, CSV), CSV)
        print("İndirildi: kagglehub")
    except Exception as e:
        print("kagglehub ile indirilemedi:", e)
if not os.path.exists(CSV):
    from google.colab import files; print("Otomatik indirme başarısız, creditcard.csv dosyasını yükleyin:"); files.upload()
df = pd.read_csv(CSV)
# Dosyanın Kaggle orijinaliyle aynı olduğunu doğrula (alt örnekler satır sırasına bağlı olduğundan önemli)
assert df.shape == (284807, 31), f"Beklenmeyen boyut: {df.shape}"
assert int(df["Class"].sum()) == 492, "Sahtekarlık sayısı 492 olmalı"
assert abs(df["Amount"].mean() - 88.3496) < 1e-3, "Amount ortalaması orijinal dosyayla uyuşmuyor"
print(f"Veri seti doğrulandı: {df.shape[0]} işlem, {int(df['Class'].sum())} sahtekarlık")
ALL["ortam"] = environment()
for k, v in ALL["ortam"].items(): print(f"{k:<14}: {v}")

def balanced_subset(n_fraud, n_normal, drop_cols):
    fraud = df[df["Class"] == 1] if n_fraud is None else df[df["Class"] == 1].sample(n=n_fraud, random_state=RANDOM_SEED)
    normal = df[df["Class"] == 0].sample(n=len(fraud) if n_normal is None else n_normal, random_state=RANDOM_SEED)
    d = pd.concat([fraud, normal]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    return d.drop(columns=[c for c in drop_cols if c in d.columns]).values.astype(float), d["Class"].values.astype(int)

# SVM / k-NN / hizalama: 100 + 100 (Time, Class çıkarılır)   | Kümeleme: 150 + 150 (Time, Class çıkarılır)
# QNN / ANN: tüm 492 sahtekarlık + 492 normal (yalnızca Class çıkarılır, 30 öznitelik)
X_svm, y_svm = balanced_subset(100, 100, ["Time", "Class"])
X_km, y_km = balanced_subset(150, 150, ["Time", "Class"])
X_nn, y_nn = balanced_subset(None, None, ["Class"])
# Anomali: AE defterindeki sıra (önce normal, sonra sahtekarlık)
_n = df[df["Class"] == 0].sample(n=400, random_state=RANDOM_SEED); _f = df[df["Class"] == 1].sample(n=100, random_state=RANDOM_SEED)
_d = pd.concat([_n, _f]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
X_ae = _d.drop(columns=["Time", "Class"]).values.astype(float); y_ae = _d["Class"].values.astype(int)
for n_, (Xa, ya) in {"SVM/k-NN": (X_svm, y_svm), "Kümeleme": (X_km, y_km), "ANN/QNN": (X_nn, y_nn), "Anomali": (X_ae, y_ae)}.items():
    print(f"{n_:<9}: {len(ya)} örnek, {Xa.shape[1]} öznitelik, sahtekarlık={ya.sum()}")


In [ ]:
FOLDS = {}
# ----- Yorum #3 / #4: sızıntısız SVM ve k-NN (yaklaşık 5–10 dk)
folds_svm, ANG_svm, KER_svm = compute_kernels(X_svm, y_svm, label="SVM/k-NN")
ALL["svm_knn"] = run_supervised(X_svm, y_svm, folds_svm, ANG_svm, KER_svm, 2, "FRAUD")
# ----- Yorum #14: hizalama (saniyeler)
ALL["kernel_hizalama"] = run_alignment(X_svm, y_svm, "FRAUD")


In [ ]:
# ----- Yorum #5 / #11: ANN, Klasik Kontrol, QNN (200 epoch, 30/100/200 kontrol noktaları; en uzun hücre)
ALL["ann_qnn"] = run_nn_block(X_nn, y_nn, 2, "FRAUD")


In [ ]:
# ----- Yorum #12: anomali tespiti, PR-AUC
ALL["anomali"] = run_autoencoders(X_ae, y_ae, "FRAUD")


In [ ]:
# ----- Yorum #13: kümeleme (Hungarian, katman içi kuantum mesafe)
ALL["kumeleme"] = run_clustering(X_km, y_km, 2, "FRAUD")


In [ ]:
# ----- Yorum #4: eşleştirilmiş t-testleri (F1) ve kayıt
S, N, C, A = ALL["svm_knn"], ALL["ann_qnn"], ALL["kumeleme"], ALL["anomali"]
nnf = lambda m, ep: [x["f1"] for x in N[m]["checkpoints"][ep]]
pairs = [("Klasik SVM (RBF)", "QSVM (Adjoint)", S["Klasik SVM (RBF)"]["f1"], S["QSVM (Adjoint Kernel)"]["f1"]),
         ("Klasik SVM (Linear)", "QSVM (Adjoint)", S["Klasik SVM (Linear)"]["f1"], S["QSVM (Adjoint Kernel)"]["f1"]),
         ("Klasik k-NN", "QKNN (Adjoint)", S["Klasik k-NN (Öklid)"]["f1"], S["QKNN (Adjoint Kernel)"]["f1"]),
         ("Saf Klasik ANN (30 ep)", "Hibrit QNN-4 (30 ep)", nnf("Saf Klasik ANN", 30), nnf("Hibrit QNN (4 Kübit)", 30)),
         ("Klasik Kontrol (30 ep)", "Hibrit QNN-4 (30 ep)", nnf("Klasik Kontrol (4 boyutlu darboğaz)", 30), nnf("Hibrit QNN (4 Kübit)", 30)),
         ("Klasik Kontrol (200 ep)", "Hibrit QNN-4 (200 ep)", nnf("Klasik Kontrol (4 boyutlu darboğaz)", 200), nnf("Hibrit QNN (4 Kübit)", 200)),
         ("Klasik K-Means", "Kuantum Çekirdek Kümeleme", C["Klasik K-Means"]["f1"], C["Kuantum Çekirdek Kümeleme"]["f1"]),
         ("Klasik K-Means", "Swap-Test Q-Means", C["Klasik K-Means"]["f1"], C["Swap-Test Q-Means"]["f1"]),
         ("Klasik AE (Derin, 30 ep)", "Kuantum QAE (30 ep)", A["Klasik AE (Derin 4-8-2-8-4)"][30]["f1"], A["Kuantum QAE (2 Katman)"][30]["f1"])]
ALL["istatistik_F1"] = paired_tests(pairs, "F1")
import shutil
os.makedirs("fraud_ek_sonuclar", exist_ok=True)
json.dump(to_py(ALL), open("fraud_ek_sonuclar/fraud_ek_sonuclar.json", "w"), ensure_ascii=False)
shutil.make_archive("fraud_ek_sonuclar", "zip", "fraud_ek_sonuclar")
print("Kaydedildi: fraud_ek_sonuclar.zip")
try:
    from google.colab import files; files.download("fraud_ek_sonuclar.zip")
except Exception: pass
